<a href="https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**My lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because I already have a working baseline for this question from ML-01. My honest, held-out result was a test-set Precision@50 = 0.620 using an 80/20 split, while an in-sample run with max_depth=3 reached 0.720 — a gap that reminds me to trust the held-out result, not the in-sample one, going forward. I don't yet have a random or naive baseline to compare 0.620 against, so I can't claim yet that the model is clearly better than doing nothing; that's something to check early in this lane. I'm picking this direction anyway because the output shape (a ranked review queue) is what I want to build toward over the next few weeks, and I already have a concrete starting point rather than starting cold.

## 2. The question: decision, action, cost of a wrong call

**Unit of analysis:** One row represents one page (`content_id`), scored using its trailing 90-day performance.

**The decision:** Which pages should a content reviewer look at first this week, out of the full page inventory, given that they only have time to review a limited number?

**The action:** A reviewer opens the top-ranked pages from my queue and decides whether to refresh, expand, protect, prune, or simply monitor each one. The ranking does not replace the reviewer's judgment; it only helps order the review queue.

**The cost of a wrong call:** If I rank a page highly and it does not actually need attention, the cost is mainly the reviewer's wasted time, which is limited but not zero. If I rank a genuinely declining page too low, the cost is potentially worse because the page may continue losing visibility or traffic without being reviewed. Therefore, missed declining pages may be more costly than unnecessary reviews, which suggests that recall may matter alongside precision when evaluating the system later.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [15]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/SNK005/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df[["impressions_90d", "clicks_90d", "ctr", "avg_position",
    "sessions_90d", "engagement_rate"]].describe()

Rows: 30000
Columns: 44


,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate
count,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,30000.000000
mean,5200.366300,16.097333,0.510733,16.34238,37.066633,2.534520
std,16838.019547,75.076958,3.279162,15.21679,107.069131,8.310096
min,1.000000,0.000000,0.000000,0.00000,1.000000,0.000000
25%,81.000000,0.000000,0.000000,6.20000,2.000000,0.000000
50%,731.000000,1.000000,0.070000,10.80000,7.000000,0.000000
75%,3615.250000,7.000000,0.290000,22.30000,27.000000,1.350000
max,517715.000000,4178.000000,100.000000,245.00000,4345.000000,100.000000


The starter dataset contains 30,000 pages, of which 16,262 (54.21%) are marked as having a downward trend. Of those declining pages, 13,152 (80.9%) still have at least 100 impressions in the last 90 days, meaning most of the decline is happening on pages that still receive measurable search visibility, not just pages with very little activity. The median age among declining pages is 216 days. This suggests a substantial, and potentially actionable, population worth investigating — not just a large but irrelevant one. These numbers do not mean every downward-trending page should be refreshed; the purpose of the ranking is to help prioritize which pages deserve human review first.

In [16]:
down = df[df["trend_direction"] == "down"]

print("Total pages:", len(df))
print("Pages with downward trend:", len(down))
print("Share of pages with downward trend:", round(len(down) / len(df) * 100, 2), "%")
print("Median age of downward-trending pages:", down["content_age_days"].median(), "days")

demand_and_declining = down[down["impressions_90d"] >= 100]

print("Declining pages with real demand (>=100 impressions):", len(demand_and_declining),
      f"({len(demand_and_declining)/len(down)*100:.1f}% of declining pages)")

Total pages: 30000
Pages with downward trend: 16262
Share of pages with downward trend: 54.21 %
Median age of downward-trending pages: 216.0 days
Declining pages with real demand (>=100 impressions): 13152 (80.9% of declining pages)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can claim:**

- Which pages, based on observed historical signals, should be prioritized for review first.
- A ranked priority list a human reviewer can use to allocate limited review time.
- Directional patterns — e.g. "pages with X and Y tend to score higher on the refresh queue" , not universal rules.
- Model performance measured honestly on held-out or otherwise appropriate validation data.

**What this work can never claim:**

- That a refresh will cause a page to recover , that would require an actual experiment (e.g. a before/after test on pages), not this dataset alone.
- That the model has learned anything about how Google's ranking algorithm actually works.
- That a low score means a page is fine , it means the page did not trigger the signals I chose to look for, which is not the same as "no problem exists."
- That in-sample results (like my earlier 0.720) represent real world performance , they should not be treated as an estimate of generalization performance; held-out or otherwise appropriate validation results are more informative.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.